# NPU-Offloaded Neural Upscaling — EDA and Implementation

Exploratory analysis of the training data, then a small upscaler trained and
exported for cross-device benchmarking.

**Scope of this notebook.** Everything device-agnostic happens here on Colab:
data analysis, training, quantization, ONNX export, and a T4 control benchmark.
The actual research measurements happen on the Snapdragon and Apple Silicon
laptops, using the ONNX file this notebook produces.

**Runtime → Change runtime type → GPU** before running.

## Contents

1. Setup and data
2. **EDA** — what the data says about model and training choices
3. Model (NPU-friendly by construction)
4. Training with distillation
5. Quantization-aware training
6. ONNX export and operator audit
7. T4 control benchmark

> **This copy matches Appendix C of the dissertation.** Three cells were
> corrected during the original run and those corrections are included here:
> the package install adds `onnxscript`, an unsupported calibration-method
> argument is removed from the quantization call, and the evaluation of the
> quantized model crops its input to the fixed exported shape.
>
> Note the dataset: this notebook trains on 90 images from the DIV2K
> **validation** split, with 10 held out. The corrected model in
> `espcn_retrain_v2.ipynb` uses the training split.


## 1. Setup and data

In [ ]:
import json
import math
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SCALE = 2
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
DATA = Path("/content/div2k")
DATA.mkdir(parents=True, exist_ok=True)
URL = "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip"

if not (DATA / "DIV2K_valid_HR").exists():
    print("downloading DIV2K valid (~450MB)...")
    zip_path = DATA / "valid.zip"
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA)
    zip_path.unlink()

images = sorted((DATA / "DIV2K_valid_HR").glob("*.png"))
train_images, eval_images = images[:-10], images[-10:]
print(f"{len(images)} images | train {len(train_images)} | held out {len(eval_images)}")

## 2. Exploratory data analysis

The EDA here is not generic image statistics. Each analysis answers a question
that changes a downstream decision:

- How much do images vary in size and shape? → patch sampling strategy
- How much high-frequency detail is there? → whether upscaling is hard at all
- What does bicubic already achieve? → the floor any model must beat
- Which images are hard, and why? → what the evaluation set should contain
- How much do patches vary within an image? → whether random cropping is wasteful

Everything below uses the training images only. The ten held-out images are not
examined, so nothing seen here can influence a modelling choice.

In [ ]:
def image_stats(path):
    """Per-image statistics, computed once and reused throughout the EDA."""
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    gray = arr.mean(axis=2)

    # Laplacian variance: a standard sharpness / high-frequency proxy. Images
    # with more fine detail are harder to upscale, because that detail is
    # exactly what downscaling destroys.
    lap = (gray[:-2, 1:-1] + gray[2:, 1:-1] +
           gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1])

    return {
        "name": path.name,
        "width": img.width,
        "height": img.height,
        "megapixels": img.width * img.height / 1e6,
        "aspect": img.width / img.height,
        "mean_r": arr[:, :, 0].mean(),
        "mean_g": arr[:, :, 1].mean(),
        "mean_b": arr[:, :, 2].mean(),
        "brightness": gray.mean(),
        "contrast": gray.std(),
        "laplacian_var": lap.var(),
    }


stats = pd.DataFrame([image_stats(p) for p in train_images])
print(stats[["width", "height", "megapixels", "brightness",
             "contrast", "laplacian_var"]].describe().round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(stats["width"], stats["height"], s=18, alpha=0.6)
axes[0].set(xlabel="Width (px)", ylabel="Height (px)",
            title="Image dimensions")
axes[0].grid(alpha=0.3)

axes[1].hist(stats["aspect"], bins=30, color="#4a7ba7", edgecolor="white")
axes[1].set(xlabel="Aspect ratio (w/h)", ylabel="Images",
            title="Aspect ratio distribution")
axes[1].grid(alpha=0.3)

axes[2].hist(stats["laplacian_var"], bins=30, color="#a7684a", edgecolor="white")
axes[2].set(xlabel="Laplacian variance", ylabel="Images",
            title="High-frequency content")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"widths: {stats['width'].min()} to {stats['width'].max()}")
print(f"heights: {stats['height'].min()} to {stats['height'].max()}")
print(f"most common width: {stats['width'].mode().iloc[0]} "
      f"({(stats['width'] == stats['width'].mode().iloc[0]).sum()} images)")
print()
print("Decision: images vary in size, so training uses fixed-size random crops")
print("rather than whole images. Batching whole images would need padding and")
print("waste compute on the padding.")

### 2.1 The bicubic floor

Bicubic upscaling is free and already quite good. Any learned model has to beat
it or it has earned nothing, so this is the number that matters most in the EDA.

It also gives a per-image difficulty measure, which the next cell uses.

In [ ]:
def bicubic_baseline(path, scale=SCALE, max_side=1024):
    """PSNR and SSIM for bicubic upscaling. The floor to beat."""
    img = Image.open(path).convert("RGB")
    w, h = min(img.width, max_side), min(img.height, max_side)
    w, h = w - w % scale, h - h % scale
    hr = img.crop((0, 0, w, h))

    lr = hr.resize((w // scale, h // scale), Image.BICUBIC)
    up = lr.resize((w, h), Image.BICUBIC)

    hr_arr = np.asarray(hr, dtype=np.float32) / 255.0
    up_arr = np.asarray(up, dtype=np.float32) / 255.0

    mse = ((hr_arr - up_arr) ** 2).mean()
    psnr = 10 * np.log10(1.0 / max(mse, 1e-12))
    return psnr


print("computing bicubic baseline (a minute or two)...")
stats["bicubic_psnr"] = [bicubic_baseline(p) for p in train_images]

print()
print(f"bicubic PSNR: mean {stats['bicubic_psnr'].mean():.2f} dB, "
      f"range {stats['bicubic_psnr'].min():.2f} to {stats['bicubic_psnr'].max():.2f}")
print()
print("Hardest images for bicubic:")
print(stats.nsmallest(5, "bicubic_psnr")[
    ["name", "bicubic_psnr", "laplacian_var"]].to_string(index=False))
print()
print("Easiest:")
print(stats.nlargest(5, "bicubic_psnr")[
    ["name", "bicubic_psnr", "laplacian_var"]].to_string(index=False))

In [ ]:
correlation = stats["laplacian_var"].corr(stats["bicubic_psnr"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(stats["laplacian_var"], stats["bicubic_psnr"], s=22, alpha=0.65)
axes[0].set(xlabel="Laplacian variance (high-frequency content)",
            ylabel="Bicubic PSNR (dB)",
            title=f"Detail against difficulty (r = {correlation:.3f})")
axes[0].grid(alpha=0.3)

axes[1].hist(stats["bicubic_psnr"], bins=30, color="#4a7ba7", edgecolor="white")
axes[1].axvline(stats["bicubic_psnr"].mean(), color="#c1440e", linestyle="--",
                label=f"mean {stats['bicubic_psnr'].mean():.1f} dB")
axes[1].set(xlabel="Bicubic PSNR (dB)", ylabel="Images",
            title="How much headroom is there?")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"correlation between high-frequency content and bicubic PSNR: {correlation:.3f}")
print()
print("Interpretation: a strong negative correlation confirms that images with")
print("more fine detail are harder to upscale, because downscaling destroys")
print("exactly that detail. It validates Laplacian variance as a difficulty")
print("proxy, which matters because the evaluation set must contain hard images")
print("or the reported gain over bicubic will be flattering and meaningless.")

### 2.2 Patch-level variation

Training samples random crops. If most crops are flat sky or wall, the model
spends its capacity learning to copy smooth regions, which bicubic already does
perfectly. This cell checks how much of the data is actually informative.

In [ ]:
def patch_variance_sample(paths, patch=128, per_image=40, seed=SEED):
    """Variance of random crops, as a proxy for how much detail each contains."""
    rng = np.random.default_rng(seed)
    rows = []
    for path in paths[:40]:  # a subset keeps this quick
        arr = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0
        h, w, _ = arr.shape
        for _ in range(per_image):
            y = rng.integers(0, h - patch)
            x = rng.integers(0, w - patch)
            crop = arr[y:y + patch, x:x + patch]
            gray = crop.mean(axis=2)
            lap = (gray[:-2, 1:-1] + gray[2:, 1:-1] +
                   gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1])
            rows.append({"std": gray.std(), "lap_var": lap.var()})
    return pd.DataFrame(rows)


patches = patch_variance_sample(train_images)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(patches["std"], bins=50, color="#4a7ba7", edgecolor="white")
axes[0].set(xlabel="Patch standard deviation", ylabel="Patches",
            title="How much variation within a crop?")
axes[0].grid(alpha=0.3)

axes[1].hist(np.log10(patches["lap_var"] + 1e-8), bins=50,
             color="#a7684a", edgecolor="white")
axes[1].set(xlabel="log10(Laplacian variance)", ylabel="Patches",
            title="Detail per crop (log scale)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

flat_threshold = patches["lap_var"].quantile(0.25)
flat_fraction = (patches["lap_var"] <= flat_threshold).mean()
print(f"{len(patches)} patches sampled")
print(f"bottom quartile of detail sits below lap_var {flat_threshold:.6f}")
print(f"fraction of near-flat patches: {flat_fraction:.1%}")
print()
print("Decision: if a large share of random crops are near-flat, rejection")
print("sampling on patch variance concentrates training on informative regions.")
print("The PatchDataset below implements this with min_variance.")

### 2.3 Channel statistics

Determines whether the model needs input normalisation beyond a simple divide
by 255, and whether the channels differ enough to matter.

In [ ]:
channel_means = stats[["mean_r", "mean_g", "mean_b"]].mean()
channel_stds = stats[["mean_r", "mean_g", "mean_b"]].std()

print("channel means across the training set:")
for channel in ["r", "g", "b"]:
    print(f"  {channel}: {channel_means[f'mean_{channel}']:.4f} "
          f"(sd across images {channel_stds[f'mean_{channel}']:.4f})")
print()
print(f"brightness: mean {stats['brightness'].mean():.4f}, "
      f"range {stats['brightness'].min():.4f} to {stats['brightness'].max():.4f}")
print()
print("Decision: channels are close enough that per-channel normalisation adds")
print("little. Scaling to [0,1] is sufficient, and it keeps the exported graph")
print("simpler -- which matters, because every extra operator is another chance")
print("of an NPU fallback.")

## 3. The model

An ESPCN-style upscaler, constrained to operators an NPU can execute.

The constraint drives the design. NPUs support a narrower operator set than GPUs,
and anything unsupported falls back to the CPU silently -- destroying both the
latency and the measurement. So: convolutions, ReLU and pixel-shuffle only. No
exotic activations, no dynamic shapes, channel counts in multiples of 8.

Pixel-shuffle upscales by rearranging channels into spatial positions, so all the
convolution work happens at low resolution. That is the single most important
efficiency decision in the architecture.

In [ ]:
class ESPCN(nn.Module):
    """
    Deliberately plain. Every operator here is well supported across ONNX
    Runtime backends, which is the point -- this is the workload guaranteed to
    run everywhere, and the control against which the transformer's expected
    fallback problems are measured.
    """

    def __init__(self, scale=SCALE, channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, channels, 5, padding=2)
        self.conv2 = nn.Conv2d(channels, channels // 2, 3, padding=1)
        self.conv3 = nn.Conv2d(channels // 2, 3 * scale * scale, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)
        self.scale = scale

    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = F.relu(self.conv2(y))
        return self.shuffle(self.conv3(y))


class TeacherSR(nn.Module):
    """
    A larger model used only to generate training targets. It never has to run
    on the NPU, so it can be as wide as compute allows.
    """

    def __init__(self, scale=SCALE, channels=128, blocks=8):
        super().__init__()
        self.head = nn.Conv2d(3, channels, 3, padding=1)
        self.body = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, 3, padding=1))
            for _ in range(blocks)
        ])
        self.tail = nn.Conv2d(channels, 3 * scale * scale, 3, padding=1)
        self.shuffle = nn.PixelShuffle(scale)

    def forward(self, x):
        feat = self.head(x)
        for block in self.body:
            feat = feat + block(feat)  # residual
        return self.shuffle(self.tail(feat))


student = ESPCN().to(DEVICE)
teacher = TeacherSR().to(DEVICE)

print(f"student: {sum(p.numel() for p in student.parameters()):,} parameters")
print(f"teacher: {sum(p.numel() for p in teacher.parameters()):,} parameters")

with torch.no_grad():
    probe = torch.randn(1, 3, 64, 64, device=DEVICE)
    print("student shape check:", tuple(probe.shape), "->", tuple(student(probe).shape))

## 4. Data pipeline and training

In [ ]:
from torch.utils.data import Dataset, DataLoader


class PatchDataset(Dataset):
    """
    Random HR crops with their bicubic-downscaled LR counterparts.

    min_variance implements the rejection sampling the EDA motivated: crops
    below the threshold are resampled, so training concentrates on regions
    where a model can actually improve on bicubic.
    """

    def __init__(self, paths, patch=128, scale=SCALE, samples=12,
                 min_variance=0.0, max_tries=10):
        self.paths = paths
        self.patch = patch
        self.scale = scale
        self.samples = samples
        self.min_variance = min_variance
        self.max_tries = max_tries
        self.cache = {}

    def __len__(self):
        return len(self.paths) * self.samples

    def _array(self, index):
        if index not in self.cache:
            self.cache[index] = np.asarray(
                Image.open(self.paths[index]).convert("RGB"), dtype=np.uint8)
        return self.cache[index]

    def __getitem__(self, i):
        arr = self._array(i // self.samples)
        h, w, _ = arr.shape

        crop = None
        for _ in range(self.max_tries):
            y = np.random.randint(0, h - self.patch)
            x = np.random.randint(0, w - self.patch)
            y -= y % self.scale
            x -= x % self.scale
            candidate = arr[y:y + self.patch, x:x + self.patch]
            if candidate.mean(axis=2).std() / 255.0 >= self.min_variance:
                crop = candidate
                break
        if crop is None:
            crop = candidate  # give up and use the last one rather than loop forever

        hr_img = Image.fromarray(crop)
        lr_img = hr_img.resize((self.patch // self.scale, self.patch // self.scale),
                               Image.BICUBIC)

        to_tensor = lambda im: torch.from_numpy(
            np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return to_tensor(lr_img), to_tensor(hr_img)


# Threshold from the EDA: skip the flattest crops.
MIN_VAR = 0.02
train_ds = PatchDataset(train_images, min_variance=MIN_VAR)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2,
                      drop_last=True)
print(f"{len(train_ds)} patches | {len(train_dl)} batches per epoch")

In [ ]:
def psnr_torch(a, b):
    mse = F.mse_loss(a.clamp(0, 1), b.clamp(0, 1))
    return (10 * torch.log10(1.0 / mse)).item()


def train_model(model, loader, epochs, lr=1e-3, teacher=None, alpha=0.5, tag=""):
    """
    Standard L1 training, optionally with distillation.

    When a teacher is supplied the loss mixes ground truth with the teacher's
    output. The teacher's predictions carry more information than the target
    images alone, which is how a model this small reaches usable quality.
    """
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(loader))
    if teacher is not None:
        teacher.eval()

    for epoch in range(1, epochs + 1):
        running, batches = 0.0, 0
        start = time.time()
        for lr_batch, hr_batch in loader:
            lr_batch = lr_batch.to(DEVICE, non_blocking=True)
            hr_batch = hr_batch.to(DEVICE, non_blocking=True)

            pred = model(lr_batch)
            loss = F.l1_loss(pred, hr_batch)

            if teacher is not None:
                with torch.no_grad():
                    soft_target = teacher(lr_batch)
                loss = (1 - alpha) * loss + alpha * F.l1_loss(pred, soft_target)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()

            running += loss.item()
            batches += 1

        print(f"{tag}epoch {epoch}/{epochs}  loss {running / batches:.4f}  "
              f"{time.time() - start:.0f}s")
    return model


print("=== teacher ===")
teacher = train_model(teacher, train_dl, epochs=8, tag="teacher ")

print()
print("=== student, distilled from teacher ===")
student = train_model(student, train_dl, epochs=12, teacher=teacher,
                      alpha=0.5, tag="student ")

## 5. Evaluation against the bicubic floor

In [ ]:
@torch.no_grad()
def evaluate(model, paths, scale=SCALE, max_side=1024, label=""):
    model.eval()
    rows = []
    for path in paths:
        img = Image.open(path).convert("RGB")
        w, h = min(img.width, max_side), min(img.height, max_side)
        w, h = w - w % scale, h - h % scale
        hr_img = img.crop((0, 0, w, h))
        lr_img = hr_img.resize((w // scale, h // scale), Image.BICUBIC)

        to_tensor = lambda im: torch.from_numpy(
            np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)[None]

        lr_t = to_tensor(lr_img).to(DEVICE)
        hr_t = to_tensor(hr_img).to(DEVICE)

        pred = model(lr_t).clamp(0, 1)
        bicubic = F.interpolate(lr_t, scale_factor=scale, mode="bicubic",
                                align_corners=False).clamp(0, 1)

        rows.append({
            "name": path.name,
            "model_psnr": psnr_torch(pred, hr_t),
            "bicubic_psnr": psnr_torch(bicubic, hr_t),
        })

    df = pd.DataFrame(rows)
    df["gain"] = df["model_psnr"] - df["bicubic_psnr"]
    print(f"{label}mean model {df['model_psnr'].mean():.3f} dB | "
          f"bicubic {df['bicubic_psnr'].mean():.3f} dB | "
          f"gain {df['gain'].mean():+.3f} dB")
    return df


eval_fp32 = evaluate(student, eval_images, label="student fp32: ")
evaluate(teacher, eval_images, label="teacher fp32: ")
BASELINE_PSNR = eval_fp32["model_psnr"].mean()

## 6. Quantization-aware training

NPUs are int8-native, so the model will be quantized whatever happens. Applying
quantization after training costs real quality on a model this small; QAT
simulates the rounding during training so the weights learn to tolerate it.

In [ ]:
from torch.ao.quantization import quantize_fx, get_default_qat_qconfig_mapping

qat_model = ESPCN()
qat_model.load_state_dict(student.state_dict())
qat_model.train()

# FBGEMM is the x86 backend, used here only to simulate int8 during training.
# The real int8 execution happens on the NPU via ONNX Runtime later.
qconfig_mapping = get_default_qat_qconfig_mapping("fbgemm")
example_input = torch.randn(1, 3, 64, 64)
qat_prepared = quantize_fx.prepare_qat_fx(qat_model, qconfig_mapping, (example_input,))
qat_prepared = qat_prepared.to(DEVICE)

print("fine-tuning with simulated quantization...")
qat_prepared = train_model(qat_prepared, train_dl, epochs=4, lr=1e-4,
                           teacher=teacher, alpha=0.5, tag="qat ")

eval_qat = evaluate(qat_prepared, eval_images, label="student QAT (simulated int8): ")
print()
print(f"quality cost of quantization: "
      f"{eval_qat['model_psnr'].mean() - BASELINE_PSNR:+.3f} dB")

## 7. ONNX export and operator audit

The single artefact carried to the other two machines. Every device runs this
same graph, so measured differences are hardware rather than reimplementation.

The operator audit matters more than it looks. Unsupported operators fall back to
the CPU silently, and a benchmark that does not check placement may be timing CPU
execution while reporting it as NPU. This cell lists what is in the graph so the
coverage check on the laptop has something to compare against.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

import onnx
import onnxruntime as ort

EXPORT_DIR = Path("/content/export")
EXPORT_DIR.mkdir(exist_ok=True)
ONNX_PATH = EXPORT_DIR / "espcn_x2.onnx"

student.eval().cpu()

# Fixed input shape. Dynamic axes are convenient but many NPU runtimes either
# reject them or fall back to CPU, so the shape is pinned to the target frame.
dummy = torch.randn(1, 3, 270, 480)

torch.onnx.export(
    student, dummy, ONNX_PATH,
    input_names=["input"], output_names=["output"],
    # Opset 13 is widely supported across QNN, OpenVINO and CoreML. Newer
    # opsets risk operators the NPU runtimes do not implement.
    opset_version=13,
    do_constant_folding=True,
    # Keep weights inside the .onnx file. Without this, large models split off a
    # sibling .onnx.data file, and moving only the .onnx to another machine
    # produces "External data path does not exist" at session creation.
    export_params=True,
)

# Fail loudly if an external data file appeared anyway.
external = list(EXPORT_DIR.glob("*.data"))
if external:
    print(f"WARNING: external data files present: {[f.name for f in external]}")
    print("Copy them alongside the .onnx, or re-export.")
else:
    print("single self-contained file, safe to move between machines")

model_onnx = onnx.load(ONNX_PATH)
onnx.checker.check_model(model_onnx)
student.to(DEVICE)

ops = sorted({node.op_type for node in model_onnx.graph.node})
print(f"exported: {ONNX_PATH}")
print(f"size: {ONNX_PATH.stat().st_size / 1024:.1f} KB")
print(f"operators in graph: {ops}")
print()
print("Carry this list to the laptop. Any operator here that the NPU runtime")
print("does not support becomes a CPU fallback, and must be reported.")

In [ ]:
# Verify the exported graph matches PyTorch before trusting it anywhere else.
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

test_input = torch.randn(1, 3, 270, 480)
with torch.no_grad():
    torch_out = student.cpu()(test_input).numpy()
onnx_out = session.run(None, {"input": test_input.numpy()})[0]
student.to(DEVICE)

max_diff = np.abs(torch_out - onnx_out).max()
print(f"max absolute difference PyTorch vs ONNX: {max_diff:.2e}")
print("PASS" if max_diff < 1e-4 else "FAIL -- export does not match, investigate")

# Metadata the benchmark scripts on the other machines will read.
meta = {
    "model": "ESPCN",
    "scale": SCALE,
    "input_shape": [1, 3, 270, 480],
    "opset": 13,
    "operators": ops,
    "parameters": int(sum(p.numel() for p in student.parameters())),
    "baseline_psnr_fp32": float(BASELINE_PSNR),
    "bicubic_psnr": float(eval_fp32["bicubic_psnr"].mean()),
    "int8_model": "espcn_x2_int8.onnx",
}
(EXPORT_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
print()
print(json.dumps(meta, indent=2))

## 7a. INT8 quantization for the NPU

**This step is required, not an optimization.** The Hexagon NPU executes integer
operations only. Handed an fp32 model, QNN accepts the session and then runs
every node on the CPU, reporting success. The symptom is a benchmark that
measures CPU execution while labelling it NPU.

Two details are not optional:

- **QDQ format.** QNN's HTP backend requires QuantFormat.QDQ, where explicit
  QuantizeLinear / DequantizeLinear nodes wrap each operation. QOperator format
  is rejected.
- **Real calibration data.** Quantization needs to observe the range of real
  activations to choose scale factors. Random noise produces ranges that do not
  match real images, and the quality loss is severe and avoidable.

Quantization runs here on Colab (x86_64) because the ONNX Runtime quantization
utilities are not supported on ARM64. The laptop only ever runs inference.

In [ ]:
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader, QuantType, QuantFormat,
)
from onnxruntime.quantization.shape_inference import quant_pre_process

INT8_PATH = EXPORT_DIR / "espcn_x2_int8.onnx"
PREPROC_PATH = EXPORT_DIR / "espcn_x2_preproc.onnx"
CALIB_SHAPE = (1, 3, 270, 480)


class ImageCalibrationReader(CalibrationDataReader):
    """
    Feeds real image crops to the quantizer.

    Scale factors are chosen from the activation ranges observed here, so the
    data must resemble what the model will actually see. Random noise gives
    ranges that no real image produces, and the quantized model is much worse
    for it.
    """

    def __init__(self, paths, input_name, n_samples=64, shape=CALIB_SHAPE):
        _, _, height, width = shape
        rng = np.random.default_rng(SEED)
        samples = []

        for i in range(n_samples):
            path = paths[i % len(paths)]
            hr = Image.open(path).convert("RGB")
            # Downscale first: the model's input is a low-resolution frame, so
            # calibration must see low-resolution statistics.
            lr = hr.resize((hr.width // SCALE, hr.height // SCALE), Image.BICUBIC)
            arr = np.asarray(lr, dtype=np.float32) / 255.0

            if arr.shape[0] < height or arr.shape[1] < width:
                arr = np.asarray(
                    lr.resize((max(width, lr.width), max(height, lr.height)),
                              Image.BICUBIC), dtype=np.float32) / 255.0

            y = rng.integers(0, arr.shape[0] - height + 1)
            x = rng.integers(0, arr.shape[1] - width + 1)
            crop = arr[y:y + height, x:x + width]
            samples.append({input_name: crop.transpose(2, 0, 1)[None].astype(np.float32)})

        self.samples = samples
        self.iterator = iter(samples)

    def get_next(self):
        return next(self.iterator, None)

    def rewind(self):
        self.iterator = iter(self.samples)


# Shape inference and graph cleanup. Skipping this is a common cause of
# quantization silently producing a model QNN then rejects.
quant_pre_process(str(ONNX_PATH), str(PREPROC_PATH), skip_symbolic_shape=True)

input_name = ort.InferenceSession(
    str(PREPROC_PATH), providers=["CPUExecutionProvider"]).get_inputs()[0].name

reader = ImageCalibrationReader(train_images, input_name, n_samples=64)
print(f"calibrating on {len(reader.samples)} real image crops")

quantize_static(
    model_input=str(PREPROC_PATH),
    model_output=str(INT8_PATH),
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,      # required by the HTP backend
    activation_type=QuantType.QUInt8,  # unsigned 8-bit, what HTP expects
    weight_type=QuantType.QUInt8,
    per_channel=False,                 # per-tensor is more widely supported on HTP
    extra_options={"ActivationSymmetric": False, "WeightSymmetric": True},
)

print(f"fp32: {ONNX_PATH.stat().st_size / 1024:6.1f} KB")
print(f"int8: {INT8_PATH.stat().st_size / 1024:6.1f} KB")

quant_model = onnx.load(INT8_PATH)
quant_ops = sorted({n.op_type for n in quant_model.graph.node})
print(f"operators after quantization: {quant_ops}")

In [ ]:
# What did quantization cost? The exported graph has a fixed input size, so
# fp32 and int8 are both evaluated on the same fixed crop for a fair comparison.
class OnnxWrapper(nn.Module):
    """Runs a fixed-shape ONNX session behind a torch-like interface."""

    def __init__(self, path, shape=(270, 480)):
        super().__init__()
        self.session = ort.InferenceSession(
            str(path), providers=["CPUExecutionProvider"])
        self.name = self.session.get_inputs()[0].name
        self.h, self.w = shape

    def eval(self):
        return self

    def forward(self, x):
        x = x[:, :, :self.h, :self.w]
        out = self.session.run(None, {self.name: x.cpu().numpy()})[0]
        return torch.from_numpy(out).to(x.device)


@torch.no_grad()
def evaluate_fixed(model, paths, in_h=270, in_w=480, label=""):
    """Evaluate on a fixed-size crop, so fp32 and int8 are compared like for like."""
    rows = []
    for path in paths:
        hr_img = Image.open(path).convert("RGB")
        lr_img = hr_img.resize((hr_img.width // SCALE, hr_img.height // SCALE),
                               Image.BICUBIC)
        to_t = lambda im: torch.from_numpy(
            np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)[None]
        lr_t = to_t(lr_img)[:, :, :in_h, :in_w].to(DEVICE)
        hr_t = to_t(hr_img)[:, :, :in_h * SCALE, :in_w * SCALE].to(DEVICE)
        pred = model(lr_t).clamp(0, 1)
        bicubic = F.interpolate(lr_t, scale_factor=SCALE, mode="bicubic",
                                align_corners=False).clamp(0, 1)
        rows.append({"name": path.name,
                     "model_psnr": psnr_torch(pred, hr_t),
                     "bicubic_psnr": psnr_torch(bicubic, hr_t)})
    df = pd.DataFrame(rows)
    df["gain"] = df["model_psnr"] - df["bicubic_psnr"]
    print(f"{label}model {df['model_psnr'].mean():.3f} dB | "
          f"bicubic {df['bicubic_psnr'].mean():.3f} dB | "
          f"gain {df['gain'].mean():+.3f} dB")
    return df


eval_fp32_crop = evaluate_fixed(student, eval_images, label="fp32 (cropped): ")
eval_int8 = evaluate_fixed(OnnxWrapper(INT8_PATH), eval_images, label="int8 QDQ:      ")

cost = eval_int8["model_psnr"].mean() - eval_fp32_crop["model_psnr"].mean()
print(f"\nquality cost of int8 quantization: {cost:+.3f} dB")
if eval_int8["gain"].mean() <= 0:
    print("!! Quantized model no longer beats bicubic — check calibration data.")

### Which file to carry to the laptop

`espcn_x2_int8.onnx` is the one that matters. The fp32 file is the control, and
useful for the CPU and GPU configurations, but the NPU will refuse it.

Download both from `/content/export`, then on the laptop:

```powershell
python check_providers.py --model espcn_x2_int8.onnx
```

The node counts should now shift from CPUExecutionProvider to QNN. If some
operators still fall back — `DepthToSpace`, the pixel-shuffle, is the likeliest
— that is a finding about operator coverage rather than a mistake, and it is
exactly what SQ2 asks about. Record which ones and carry on.

## 8. T4 control benchmark

The discrete-GPU, no-NPU baseline. This is the configuration every published
paper measures, and the control against which the laptop results are compared.

The two timings reported separately are the important part. `inference_ms` is
what benchmarks usually report. `end_to_end_ms` includes getting the frame in and
the result out. On discrete hardware the gap between them is PCIe transfer -- and
the hypothesis is that it shrinks or vanishes on unified memory.

In [ ]:
def benchmark_ort(onnx_path, provider, shape, warmup=10, iters=50):
    """Latency for one provider, separating inference from end-to-end."""
    session = ort.InferenceSession(str(onnx_path), providers=[provider])
    name = session.get_inputs()[0].name
    data = np.random.randn(*shape).astype(np.float32)

    for _ in range(warmup):
        session.run(None, {name: data})

    inference, end_to_end = [], []
    for _ in range(iters):
        start = time.perf_counter()
        fresh = np.random.randn(*shape).astype(np.float32)  # simulate a new frame
        mid = time.perf_counter()
        session.run(None, {name: fresh})
        stop = time.perf_counter()
        inference.append((stop - mid) * 1000)
        end_to_end.append((stop - start) * 1000)

    median = lambda v: sorted(v)[len(v) // 2]
    return {
        "provider": provider,
        "inference_ms": round(median(inference), 3),
        "end_to_end_ms": round(median(end_to_end), 3),
        "iqr_ms": round(np.percentile(inference, 75) - np.percentile(inference, 25), 3),
    }


shape = (1, 3, 270, 480)
available = ort.get_available_providers()
print("available providers:", available)

rows = []
for provider in ["CPUExecutionProvider", "CUDAExecutionProvider"]:
    if provider in available:
        rows.append(benchmark_ort(ONNX_PATH, provider, shape))

control = pd.DataFrame(rows)
print()
print(control.to_string(index=False))
control.to_csv(EXPORT_DIR / "t4_control_benchmark.csv", index=False)

print()
print("iqr_ms is the interquartile range: the run-to-run noise floor. A")
print("difference smaller than this between two configurations is not a result.")

In [ ]:
torch.save({
    "student_state": student.state_dict(),
    "teacher_state": teacher.state_dict(),
    "baseline_psnr": BASELINE_PSNR,
    "min_variance": MIN_VAR,
    "scale": SCALE,
}, EXPORT_DIR / "checkpoints.pt")

stats.to_csv(EXPORT_DIR / "eda_image_stats.csv", index=False)
eval_fp32.to_csv(EXPORT_DIR / "eval_fp32.csv", index=False)

print("saved to /content/export:")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")
print()
print("Download espcn_x2.onnx and model_meta.json, then run the benchmark")
print("script on the Snapdragon and Mac. Those measurements are the research.")

## Next steps

This notebook produced the artefact. The research measurements happen elsewhere.

**On the Snapdragon laptop:** install `onnxruntime-qnn`, load `espcn_x2.onnx`,
and before timing anything, verify which operators actually execute on the
Hexagon NPU. Silent CPU fallback is the main threat to the validity of every
number that follows.

**On the Mac:** CoreML execution provider, same graph, same procedure.

**Both:** report `inference_ms` and `end_to_end_ms` separately, and quote the
interquartile range alongside every median. A speedup smaller than the noise
floor is not a speedup.